# `06-workflow.ipynb`

In [2]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

llm = init_chat_model('gpt-4.1-mini')

## Prompt Chaining

- 매우 잘 정리된 업무 순서가 있을 경우 사용
- 이전 노드에서 처리한 내용을 `state` 에 담아 다음 노드로 전송

In [7]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
# 그림 보는 용
from IPython.display import Image, display


# Graph State
class State(TypedDict):
    topic: str
    joke: str
    improved_joke: str
    final_joke: str


# Nodes
def generate_joke(state: State):
    msg = llm.invoke(f'주제 {state['topic']}에 관련된 짧은 농담 생성')
    return {'joke': msg.content}


def improve_joke(state: State):
    msg = llm.invoke(f'말장난을 추가해서 아래 농담을 더 재밌게 만들어보자.\n {state['joke']}')
    return {'improved_joke': msg.content}


def polish_joke(state: State):
    msg = llm.invoke(f'아래 농담을 쩔게 뒤틀어 보자. \n {state['improved_joke']}')
    return {'final_joke': msg.content}


# Router
def check_punchline(state: State):
    # ? 나 ! 없으면, 농담을 improve 하고, 있으면 그대로 진행
    if '?' in state['joke'] or '!' in state['joke']:
        return 'Pass'
    else:
        return 'Fail'
    

workflow = StateGraph(State)

# 조립
workflow.add_node(generate_joke)
workflow.add_node(improve_joke)
workflow.add_node(polish_joke)

workflow.add_edge(START, 'generate_joke')
workflow.add_conditional_edges(
    'generate_joke',
    check_punchline,
    {
        'Pass': END,
        'Fali': 'improve_joke'
    }
)
workflow.add_edge('improve_joke', 'polish_joke')
workflow.add_edge('polish_joke', END)

graph = workflow.compile()
graph

graph.invoke({'topic': '랭체인'})

{'topic': '랭체인',
 'joke': '랭체인 개발자: "내 코드가 체인처럼 단단해!"  \n친구: "그래? 디버깅할 때는 안 풀리던데?"'}